# 03 Final Results and Dashboard

This notebook collects the saved test results and final plots into one teacher-friendly view. It does not train or tune the model. Run notebooks 00, 01, and 02 first.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from IPython.display import display

PROJECT_DIR=Path.cwd()
processed=pd.read_csv(PROJECT_DIR / "data" / "processed" / "strike_model_table.csv",low_memory=False)
metrics=pd.read_csv(PROJECT_DIR / "outputs" / "test_metrics.csv")
predictions=pd.read_csv(PROJECT_DIR / "outputs" / "test_predictions.csv")
print("Final model:",metrics.loc[0,"model"])
display(metrics)

In [ ]:
sns.set_theme(style="whitegrid")
fig,axes=plt.subplots(1,2,figsize=(13,4.5))

year_counts=processed.groupby("INCIDENT_YEAR").size()
axes[0].plot(year_counts.index,year_counts.values,color="#246b7a",linewidth=2)
axes[0].set(title="Reported wildlife strikes by year",xlabel="Incident year",ylabel="Reports")

damage_rates=(processed.groupby("PHASE_OF_FLT")["AIRCRAFT_DAMAGED"].agg(["size","mean"])
              .query("size >= 50").sort_values("mean"))
axes[1].barh(damage_rates.index,damage_rates["mean"]*100,color="#dc8750")
axes[1].set(title="Reported damage share by flight phase",xlabel="Damage reports (%)",ylabel="Flight phase")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "outputs" / "figures" / "final_overview.png",dpi=160)
plt.show()

In [ ]:
cm=confusion_matrix(predictions["actual_damage"],predictions["predicted_damage"],labels=[0,1])
fig,ax=plt.subplots(figsize=(5.5,4.5))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,
            xticklabels=["No damage","Damage"],yticklabels=["No damage","Damage"],ax=ax)
ax.set(title="2025 test set confusion matrix",xlabel="Predicted label",ylabel="Actual label")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "outputs" / "figures" / "test_confusion_matrix.png",dpi=160)
plt.show()

print("The model predicts damage among known strike reports only.")
print("The FAA database does not include all flights without a strike; these graphs are reported counts/shares, not per-flight strike risk.")

## Final presentation order
1. State the question and clarify that the dataset contains reported strike incidents.
2. Show the dataset profile and explain the damage label.
3. Explain the small feature set and how missingness, categories, skew, outliers, scaling, and leakage were handled.
4. Compare the majority baseline, Logistic Regression, and Random Forest on validation data.
5. Show 2025 holdout metrics and confusion matrix, then explain one strength and one limitation.
6. Use the year/phase graphs to show the descriptive analysis.

Avoid saying the model predicts whether a flight will strike wildlife. It predicts the reported damage label conditional on a report.